# Stardew Valley Pixel Art LoRA Trainer
**SD 1.5 · T4 GPU · Colab 2026**

**Run order:** Cell 0 → **Runtime > Restart Runtime** → Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5

Dataset ZIP: `name.png` + `name.txt` pairs (flat or one subfolder level).

In [1]:
# Cell 0: Install dependencies
# Run once, then Runtime > Restart Runtime before continuing.
#
# Fix log:
#   huggingface_hub must be locked to 0.21.4 BEFORE diffusers,
#   otherwise diffusers raises ImportError on cached_download.
#   --no-deps on diffusers prevents pip from upgrading hub back.
#   numpy pinned to 1.26.4 to fix torch.from_numpy ABI mismatch with numpy 2.x.

import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('[WARN]', r.stderr[-400:])

print('[1/4] huggingface_hub==0.21.4 (must be locked first)')
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--force-reinstall',
     'huggingface_hub==0.21.4'])

print('[2/4] diffusers / transformers / peft / safetensors')
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps',
     'diffusers==0.27.2', 'transformers==4.39.3', 'accelerate==0.29.3',
     'peft==0.10.0', 'safetensors==0.4.3', 'tokenizers==0.15.2',
     'einops', 'omegaconf'])

print('[3/4] numpy==1.26.4 (fixes binary incompatibility with torch 2.2.x)')
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--force-reinstall',
     'numpy==1.26.4'])

print('[4/4] xformers (optional, skipped if install fails)')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    'xformers==0.0.25.post1'], capture_output=True)
print('  xformers:', 'OK' if r.returncode == 0 else 'skipped (attention slicing will be used)')

print('\nDone. --> Runtime > Restart Runtime, then run Cell 1 onwards.')

[1/4] huggingface_hub==0.21.4 (must be locked first)
[2/4] diffusers / transformers / peft / safetensors
[3/4] numpy==1.26.4 (fixes binary incompatibility with torch 2.2.x)
[4/4] xformers (optional, skipped if install fails)
  xformers: skipped (attention slicing will be used)

Done. --> Runtime > Restart Runtime, then run Cell 1 onwards.


In [1]:
# Cell 1: Upload dataset ZIP
import os, zipfile, shutil
from google.colab import files

DATASET_DIR = '/content/lora_dataset'
os.makedirs(DATASET_DIR, exist_ok=True)

print('Upload your dataset ZIP (image + .txt caption pairs)...')
uploaded = files.upload()

for filename, data in uploaded.items():
    zip_path = f'/content/{filename}'
    with open(zip_path, 'wb') as f:
        f.write(data)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/content/raw_upload')

# Flatten subfolders into DATASET_DIR
for root, _, flist in os.walk('/content/raw_upload'):
    for fname in flist:
        if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.txt')):
            shutil.copy2(os.path.join(root, fname), os.path.join(DATASET_DIR, fname))

imgs = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith(('.png','.jpg','.jpeg','.webp'))]
txts = [f for f in os.listdir(DATASET_DIR) if f.endswith('.txt')]
print(f'Images: {len(imgs)} | Captions: {len(txts)}')
if len(imgs) != len(txts):
    print('WARNING: image/caption count mismatch — check your ZIP.')
if txts:
    with open(os.path.join(DATASET_DIR, txts[0])) as f:
        print(f'Sample caption: {f.read().strip()[:120]}')

Upload your dataset ZIP (image + .txt caption pairs)...


Saving train.zip to train.zip
Images: 30 | Captions: 30
Sample caption: stardew_pixel_art, Lewis, 1boy, solo, portrait, grey hair, grey mustache, facial hair, brown flat cap, hat, green shirt,


In [2]:
# Cell 2: Config
import os

CONFIG = {
    'base_model_id'      : 'runwayml/stable-diffusion-v1-5',
    'dataset_dir'        : '/content/lora_dataset',
    'output_dir'         : '/content/lora_output',

    # Training
    'resolution'         : 512,
    'batch_size'         : 2,
    'num_epochs'         : 30,        # 30 images / batch 2 = 15 steps/epoch -> 450 total
    'learning_rate'      : 5e-5,
    'lr_scheduler'       : 'cosine',  # 'cosine' | 'linear' | 'constant'
    'lr_warmup_steps'    : 50,

    # LoRA
    'lora_rank'          : 16,
    'lora_alpha'         : 16,
    'lora_dropout'       : 0.05,
    'lora_target_modules': ['to_q', 'to_k', 'to_v', 'to_out.0', 'proj_in', 'proj_out'],

    # Precision & memory
    'mixed_precision'        : 'fp16',
    'gradient_checkpointing' : True,

    'save_every_n_epochs': 10,  # intermediate checkpoint frequency

    # Must match the trigger word in your .txt caption files
    'trigger_word'       : 'stardew_pixel_art',
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)

n_imgs          = len([f for f in os.listdir(CONFIG['dataset_dir'])
                       if f.lower().endswith(('.png','.jpg','.jpeg','.webp'))])
steps_per_epoch = max(1, n_imgs // CONFIG['batch_size'])
total_steps     = steps_per_epoch * CONFIG['num_epochs']

print(f'Images: {n_imgs} | Steps/epoch: {steps_per_epoch} | Total steps: {total_steps}')
print(f'LoRA rank={CONFIG["lora_rank"]} alpha={CONFIG["lora_alpha"]} | '
      f'LR={CONFIG["learning_rate"]} ({CONFIG["lr_scheduler"]})')
print(f'Trigger word: [{CONFIG["trigger_word"]}]')

Images: 30 | Steps/epoch: 15 | Total steps: 450
LoRA rank=16 alpha=16 | LR=5e-05 (cosine)
Trigger word: [stardew_pixel_art]


In [3]:
# Cell 3: Train
#
# Key decisions:
#   - Pure PIL image pipeline (no torchvision) -> avoids numpy ABI conflict
#   - num_workers=0 -> prevents DataLoader fork/tokenizer deadlock
#   - LoRA params cast to fp32 after get_peft_model -> fixes GradScaler unscale error
#   - No accelerate launch -> avoids multiprocess deadlock
#   - scaler.step() only, no unscale_+clip_grad -> avoids fp16 gradient ValueError

import os, gc, math, time, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import numpy as np

from diffusers import DDPMScheduler, AutoencoderKL, UNet2DConditionModel
from transformers import CLIPTokenizer, CLIPTextModel
from peft import LoraConfig, get_peft_model
from safetensors.torch import save_file

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE  = torch.float16 if CONFIG['mixed_precision'] == 'fp16' else torch.float32
print(f'Device: {DEVICE} | dtype: {CONFIG["mixed_precision"]}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} | '
          f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

USE_XFORMERS = False
try:
    import xformers.ops
    USE_XFORMERS = True
    print('xformers: enabled')
except ImportError:
    print('xformers: not available, using attention slicing')


# ── Dataset (pure PIL, no torchvision) ───────────────────────────
class LoRADataset(Dataset):
    IMG_EXTS = ('.png', '.jpg', '.jpeg', '.webp')

    def __init__(self, dataset_dir, tokenizer, resolution, trigger_word=''):
        self.tokenizer  = tokenizer
        self.resolution = resolution
        self.image_paths = sorted([
            os.path.join(dataset_dir, f) for f in os.listdir(dataset_dir)
            if f.lower().endswith(self.IMG_EXTS)
        ])
        if not self.image_paths:
            raise ValueError(f'No images found in {dataset_dir}')

        self.captions = []
        for img_path in self.image_paths:
            txt = os.path.splitext(img_path)[0] + '.txt'
            caption = open(txt).read().strip() if os.path.exists(txt) else ''
            if trigger_word and trigger_word not in caption:
                caption = f'{trigger_word}, {caption}' if caption else trigger_word
            self.captions.append(caption)

    def __len__(self):
        return len(self.image_paths)

    def _load_image(self, path):
        img  = Image.open(path).convert('RGB')
        w, h = img.size
        scale = self.resolution / min(w, h)
        img   = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        nw, nh = img.size
        img   = img.crop(((nw - self.resolution) // 2, (nh - self.resolution) // 2,
                           (nw + self.resolution) // 2, (nh + self.resolution) // 2))
        arr = np.array(img, dtype=np.float32) / 255.0
        arr = (arr - 0.5) / 0.5  # normalize to [-1, 1]
        return torch.from_numpy(arr).permute(2, 0, 1)  # [C, H, W]

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.captions[idx],
            max_length=self.tokenizer.model_max_length,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'pixel_values'  : self._load_image(self.image_paths[idx]),
            'input_ids'     : enc.input_ids.squeeze(0),
            'attention_mask': enc.attention_mask.squeeze(0),
        }


# ── Load model components ─────────────────────────────────────────
MODEL_ID = CONFIG['base_model_id']
print('Loading tokenizer...')
tokenizer    = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder='tokenizer')
print('Loading text encoder...')
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder='text_encoder')
text_encoder = text_encoder.to(DEVICE, dtype=DTYPE)
text_encoder.requires_grad_(False)
print('Loading VAE...')
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder='vae')
vae = vae.to(DEVICE, dtype=DTYPE)
vae.requires_grad_(False)
print('Loading UNet...')
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder='unet')
unet = unet.to(DEVICE, dtype=DTYPE)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder='scheduler')
print('All components loaded.')


# ── Inject LoRA ───────────────────────────────────────────────────
unet = get_peft_model(unet, LoraConfig(
    r=CONFIG['lora_rank'], lora_alpha=CONFIG['lora_alpha'],
    target_modules=CONFIG['lora_target_modules'],
    lora_dropout=CONFIG['lora_dropout'], bias='none',
))

# Cast LoRA trainable params to fp32 to prevent GradScaler unscale ValueError
for _, param in unet.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

if CONFIG['gradient_checkpointing']:
    unet.enable_gradient_checkpointing()
if USE_XFORMERS:
    unet.enable_xformers_memory_efficient_attention()
elif hasattr(unet, 'enable_attention_slicing'):
    unet.enable_attention_slicing()

trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in unet.parameters())
print(f'Trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.2f}%)')


# ── DataLoader ────────────────────────────────────────────────────
dataset    = LoRADataset(CONFIG['dataset_dir'], tokenizer,
                         CONFIG['resolution'], CONFIG['trigger_word'])
dataloader = DataLoader(dataset, batch_size=CONFIG['batch_size'],
                        shuffle=True, num_workers=0,  # 0 prevents fork deadlock
                        pin_memory=True, drop_last=True)
print(f'Dataset: {len(dataset)} images | {len(dataloader)} steps/epoch')


# ── Optimizer & LR schedule ───────────────────────────────────────
optimizer = torch.optim.AdamW(
    [p for p in unet.parameters() if p.requires_grad],
    lr=CONFIG['learning_rate'], betas=(0.9, 0.999), weight_decay=1e-2, eps=1e-8,
)

num_epochs         = CONFIG['num_epochs']
num_training_steps = num_epochs * len(dataloader)

def get_lr(step, total, warmup, base, sched):
    if step < warmup:
        return base * step / max(1, warmup)
    p = (step - warmup) / max(1, total - warmup)
    if sched == 'cosine': return base * 0.5 * (1 + math.cos(math.pi * p))
    if sched == 'linear': return base * (1 - p)
    return base

scaler = torch.cuda.amp.GradScaler(enabled=(CONFIG['mixed_precision'] == 'fp16'))


# ── Training loop ─────────────────────────────────────────────────
print(f'\nStarting: {num_epochs} epochs / {num_training_steps} total steps')
unet.train()
global_step  = 0
train_start  = time.time()
loss_history = []

for epoch in range(1, num_epochs + 1):
    epoch_loss = 0.0
    pbar = tqdm(dataloader, desc=f'Epoch {epoch}/{num_epochs}', leave=True)

    for batch in pbar:
        lr = get_lr(global_step, num_training_steps, CONFIG['lr_warmup_steps'],
                    CONFIG['learning_rate'], CONFIG['lr_scheduler'])
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        pixel_values   = batch['pixel_values'].to(DEVICE, dtype=DTYPE)
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)

        with torch.cuda.amp.autocast(enabled=(CONFIG['mixed_precision'] == 'fp16')):
            with torch.no_grad():
                latents = vae.encode(pixel_values).latent_dist.sample()
                latents = latents * vae.config.scaling_factor
                encoder_hidden_states = text_encoder(
                    input_ids, attention_mask=attention_mask).last_hidden_state

            noise     = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps,
                (latents.shape[0],), device=DEVICE).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
            noise_pred    = unet(noisy_latents, timesteps,
                                 encoder_hidden_states=encoder_hidden_states).sample
            loss = F.mse_loss(noise_pred.float(), noise.float(), reduction='mean')

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)   # no unscale_+clip_grad to avoid fp16 gradient error
        scaler.update()

        loss_val = loss.item()
        epoch_loss  += loss_val
        loss_history.append(loss_val)
        global_step += 1
        pbar.set_postfix({'loss': f'{loss_val:.4f}', 'lr': f'{lr:.2e}', 'step': global_step})

    pbar.close()
    avg = epoch_loss / len(dataloader)
    print(f'  Epoch {epoch:3d}/{num_epochs} | avg_loss={avg:.4f} | step={global_step}')

    if epoch % CONFIG['save_every_n_epochs'] == 0:
        ckpt = os.path.join(CONFIG['output_dir'], f'ckpt_epoch{epoch:03d}.safetensors')
        save_file({n: p.data.cpu().float()
                   for n, p in unet.named_parameters() if p.requires_grad}, ckpt)
        print(f'  Checkpoint saved: {ckpt}')

print(f'\nFinished in {(time.time()-train_start)/60:.1f} min | final loss: {loss_history[-1]:.4f}')


# ── Save final weights ────────────────────────────────────────────
final_path = os.path.join(CONFIG['output_dir'], 'stardew_pixel_lora_final.safetensors')
save_file({n: p.data.cpu().float() for n, p in unet.named_parameters() if p.requires_grad},
          final_path)

if os.path.exists(final_path):
    print(f'Saved: {final_path} ({os.path.getsize(final_path)/1e6:.1f} MB)')
else:
    print('ERROR: file not saved — check disk space.')

try:
    unet.save_pretrained(os.path.join(CONFIG['output_dir'], 'peft_format'))
    print('PEFT format also saved.')
except Exception as e:
    print(f'PEFT format skipped: {e}')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


Device: cuda | dtype: fp16
GPU: Tesla T4 | VRAM: 15.6 GB
xformers: not available, using attention slicing
Loading tokenizer...


Loading text encoder...


Loading VAE...


Loading UNet...


All components loaded.
Trainable: 3,987,456 / 863,508,420 (0.46%)
Dataset: 30 images | 15 steps/epoch

Starting: 30 epochs / 450 total steps


... (26 outputs left out)

  Epoch  27/30 | avg_loss=0.0703 | step=405


  Epoch  28/30 | avg_loss=0.0732 | step=420


  Epoch  29/30 | avg_loss=0.0539 | step=435


  Epoch  30/30 | avg_loss=0.0667 | step=450
  Checkpoint saved: /content/lora_output/ckpt_epoch030.safetensors

Finished in 7.1 min | final loss: 0.1213
Saved: /content/lora_output/stardew_pixel_lora_final.safetensors (16.0 MB)
PEFT format also saved.


In [4]:
# Cell 4: Download weights
from google.colab import files
import os

path = os.path.join(CONFIG['output_dir'], 'stardew_pixel_lora_final.safetensors')
if os.path.exists(path):
    files.download(path)
else:
    print('File not found — run Cell 3 first.')
    for f in os.listdir(CONFIG['output_dir']):
        print(' ', f)

In [16]:
# Cell 5: Inference — verify LoRA weights are working
#
# Manual merge: W_new = W_orig + scale * (lora_B @ lora_A)
# .view() handles both 2D linear and 4D conv1x1 weight shapes.
# All tensors kept on CPU to avoid device mismatch errors.

import torch, gc, os
from diffusers import StableDiffusionPipeline
from safetensors.torch import load_file
from IPython.display import display

try:
    del vae, text_encoder
    gc.collect(); torch.cuda.empty_cache()
except: pass

pipe = StableDiffusionPipeline.from_pretrained(
    CONFIG['base_model_id'], torch_dtype=torch.float16, safety_checker=None).to('cuda')

lora_weights = load_file(os.path.join(CONFIG['output_dir'], 'stardew_pixel_lora_final.safetensors'))
LORA_SCALE   = 0.8  # adjust between 0.5 and 1.0

unet_sd = pipe.unet.state_dict()
merged, skipped = 0, 0

for a_key in [k for k in lora_weights if k.endswith('lora_A.default.weight')]:
    b_key    = a_key.replace('lora_A', 'lora_B')
    base_key = a_key.replace('base_model.model.', '').replace('.lora_A.default.weight', '.weight')
    if b_key not in lora_weights or base_key not in unet_sd:
        skipped += 1; continue

    lora_A = lora_weights[a_key].float().cpu()
    lora_B = lora_weights[b_key].float().cpu()
    orig   = unet_sd[base_key].float().cpu()

    A_2d = lora_A.view(lora_A.shape[0], -1)
    B_2d = lora_B.view(lora_B.shape[0], -1)
    unet_sd[base_key] = orig + (B_2d @ A_2d).view(orig.shape) * LORA_SCALE
    merged += 1

pipe.unet.load_state_dict(unet_sd, strict=True)
pipe.unet = pipe.unet.half()
print(f'LoRA merged: {merged} weights applied, {skipped} skipped')

PROMPT = (
    f"{CONFIG['trigger_word']}, 1girl, solo, pure white background, upper body, facing viewer, smile, long hair"
)
NEG = "scenery, landscape, blurry, realistic, multiple characters"
print(f'Prompt: {PROMPT}')

with torch.autocast('cuda'):
    image = pipe(PROMPT, negative_prompt=NEG,
                 num_inference_steps=30, guidance_scale=7.5,
                 width=512, height=512).images[0]

display(image)
out = os.path.join(CONFIG['output_dir'], 'test_generation.png')
image.save(out)
print(f'Saved: {out}')

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


LoRA merged: 160 weights applied, 0 skipped
Prompt: stardew_pixel_art, 1girl, solo, pure white background, upper body, facing viewer, smile, long hair


Saved: /content/lora_output/test_generation.png
